In [0]:
%sql

SELECT COUNT(*) FROM bootcamp.silver.propiedades;

In [0]:
%sql

DESCRIBE TABLE bootcamp.silver.propiedades;

#Modelado Gold - Star Schema

## Modulo 1-E1.1

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_zona;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona (
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1)  COMMENT "PK-Identificador único de la zona",
    partido STRING NOT NULL COMMENT "Partido donde se encuentra la propiedad",
    region STRING NOT NULL COMMENT "Region donde se encuentra la propiedad",
    ciudad STRING NOT NULL COMMENT "Ciudad donde se encuentra la propiedad",
    provincia STRING NOT NULL COMMENT "Provincia donde se encuentra la propiedad" DEFAULT "Buenos Aires",
    pais string NOT NULL COMMENT "Pais donde se encuentra la propiedad" DEFAULT "Argentina",
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Zona";



In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.gold.dim_zona;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona (
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1)  COMMENT "PK-Identificador único de la zona",
    partido STRING NOT NULL COMMENT "Partido donde se encuentra la propiedad",
    region STRING NOT NULL COMMENT "Region donde se encuentra la propiedad",
    ciudad STRING NOT NULL COMMENT "Ciudad donde se encuentra la propiedad",
    provincia STRING NOT NULL COMMENT "Provincia donde se encuentra la propiedad" DEFAULT "Buenos Aires",
    pais string NOT NULL COMMENT "Pais donde se encuentra la propiedad" DEFAULT "Argentina",
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY(zona_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Zona";

In [0]:
%sql

DESCRIBE bootcamp.gold.dim_zona;

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_tipo_operacion;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tipo_operacion(
    tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1)  COMMENT "PK-Identificador único de la operacion",
    tipo_operacion STRING NOT NULL COMMENT "Tipo de operacion",
    moneda STRING NOT NULL COMMENT "Tipo de moneda",
    categoria STRING NOT NULL COMMENT "Categoria",
    descripcion STRING NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY(tipo_operacion_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Tipo Operacion";

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_tiempo;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tiempo(
    fecha_id BIGINT COMMENT "PK-Identificador único de la fecha",
    fecha STRING NOT NULL,
    anio STRING NOT NULL,
    mes STRING NOT NULL,
    trimestre STRING NOT NULL,
    dia_semana STRING NOT NULL,
    es_fin_de_semana BOOLEAN NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY(fecha_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Estatica Tiempo";

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_caracteristicas;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_caracteristicas(
    caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT "PK-Identificador único de las caracteristicas",
    estado STRING NOT NULL,
    cochera BOOLEAN NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY(caracteristicas_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Caracteristicas";


In [0]:
DROP TABLE IF EXISTS bootcamp.gold.dim_orientacion;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_orientacion(
    orientacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT "PK-Identificador único de la orientacion",
    orientacion STRING NOT NULL,
    tipo_orientacion STRING NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY(orientacion_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Orientacion";

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.fact_propiedades;

CREATE TABLE IF NOT EXISTS bootcamp.gold.fact_propiedades(
    row_hash STRING NOT NULL COMMENT "PK-ROW HASH único de la propiedad",
    zona_id BIGINT NOT NULL COMMENT "FK - dim_zona",
    tipo_orientacion_id BIGINT NOT NULL COMMENT "FK-dim_orientacion",
    fecha_id BIGINT NOT NULL COMMENT "FK-dim_tiempo",
    caracteristicas_id BIGINT NOT NULL COMMENT "FK-dim_caracteristicas",
    tipo_operacion_id BIGINT NOT NULL COMMENT "FK-dim_tipo_operacion",
    url STRING NOT NULL COMMENT "URL de la propiedad",
    precio DECIMAL(15,2),
    expensas DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    ambientes INT,    
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY(row_hash),   
    FOREIGN KEY (zona_id)
    REFERENCES bootcamp.gold.dim_zona (zona_id),
    FOREIGN KEY (tipo_orientacion_id)
    REFERENCES bootcamp.gold.dim_orientacion (orientacion_id),
    FOREIGN KEY (fecha_id)
    REFERENCES bootcamp.gold.dim_tiempo (fecha_id),
    FOREIGN KEY (caracteristicas_id)
    REFERENCES bootcamp.gold.dim_caracteristicas (caracteristicas_id),
    FOREIGN KEY (tipo_operacion_id)
    REFERENCES bootcamp.gold.dim_tipo_operacion (tipo_operacion_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Fact Propiedades";

In [0]:
%sql
INSERT INTO bootcamp.gold.fact_propiedades (
    row_hash,
    zona_id,
    tipo_orientacion_id,
    fecha_id,
    caracteristicas_id,
    tipo_operacion_id,
    url
)
VALUES (
    'abc',
    9999,
    1,
    20260101,
    1,
    1,
    'https://test.com'
);